# SSR Crawler
Crawl web tinh (requests + BeautifulSoup)

In [9]:
from google.colab import drive
import os
drive.mount('/content/drive')
WORK_DIR = '/content/drive/MyDrive/Crawl_Data/CrawlData'
DOWNLOAD_DIR = os.path.join(WORK_DIR, 'downloaded_files')
os.makedirs(DOWNLOAD_DIR, exist_ok=True)
os.chdir(WORK_DIR)
print('Work:', WORK_DIR)

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Work: /content/drive/MyDrive/Crawl_Data/CrawlData


In [10]:
!pip install requests beautifulsoup4 lxml -q

In [11]:
import requests
from bs4 import BeautifulSoup
import json, time, re
from urllib.parse import urljoin, urlparse, unquote
from datetime import datetime
from concurrent.futures import ThreadPoolExecutor, as_completed

INPUT_FILE = os.path.join(WORK_DIR, 'ssr_urls.json')
OUTPUT_FILE = os.path.join(WORK_DIR, 'output_ssr.json')
HEADERS = {'User-Agent': 'Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36'}

In [12]:
def get_filename(url):
    path = urlparse(url).path
    filename = unquote(os.path.basename(path))
    if not filename or filename == '/': filename = f'file_{hash(url) % 100000}'
    if '.' not in filename: filename = filename + '.bin'
    return filename

def is_downloadable(url, exts):
    path = urlparse(url.lower()).path
    return any(path.endswith(e) for e in exts)

def is_same_domain(url, base):
    return urlparse(base).netloc == urlparse(url).netloc

def extract_links(html, base, selector=None, exts=None, require_ext=False):
    soup = BeautifulSoup(html, 'lxml')
    links, seen = [], set()
    if selector and selector.get('selector') and selector.get('type') == 'css':
        for elem in soup.select(selector['selector']):
            if elem.name == 'a' and elem.get('href'):
                url = urljoin(base, elem['href'])
                if url not in seen: seen.add(url); links.append(url)
            for a in elem.find_all('a', href=True):
                url = urljoin(base, a['href'])
                if url not in seen: seen.add(url); links.append(url)
    else:
        for a in soup.find_all('a', href=True):
            url = urljoin(base, a['href'])
            if url not in seen:
                if require_ext and exts:
                    if is_downloadable(url, exts): seen.add(url); links.append(url)
                else: seen.add(url); links.append(url)
    return links

In [13]:
def download(url, folder, filename=None):
    result = {'url': url, 'filename': filename or get_filename(url), 'status': 'pending'}
    try:
        r = requests.get(url, headers=HEADERS, timeout=120, stream=True, allow_redirects=True)
        r.raise_for_status()
        if 'Content-Disposition' in r.headers:
            cd = r.headers['Content-Disposition']
            for p in [r"filename\*=UTF-8''(.+)", r'filename="(.+)"', r"filename='(.+)'", r'filename=([^;\s]+)']:
                m = re.search(p, cd)
                if m: result['filename'] = unquote(m.group(1).strip()); break
        path = os.path.join(folder, result['filename'])
        c = 1; base_path, ext = os.path.splitext(path)
        while os.path.exists(path): path = f'{base_path}_{c}{ext}'; c += 1
        with open(path, 'wb') as f:
            for chunk in r.iter_content(8192): f.write(chunk)
        result['status'] = 'success'
        result['size'] = os.path.getsize(path)
        result['path'] = path
    except Exception as e:
        result['status'] = 'failed'
        result['error'] = str(e)
    return result

In [14]:
def crawl_level(urls, level_config, base_domain, delay, exts):
    selector = level_config.get('selector')
    max_pages = level_config.get('max_pages', 100)
    is_download = level_config.get('is_download', False)
    all_links, crawled_pages = [], []
    urls = urls[:max_pages]
    for i, url in enumerate(urls, 1):
        print(f'  [{i}/{len(urls)}] {url[:55]}...')
        try:
            time.sleep(delay)
            r = requests.get(url, headers=HEADERS, timeout=30)
            r.raise_for_status()
            links = extract_links(r.text, url, selector, exts, require_ext=is_download and not selector)
            if not is_download:
                links = [l for l in links if is_same_domain(l, base_domain) and l != url]
            print(f'      -> {len(links)} links')
            crawled_pages.append({'url': url, 'links_found': len(links), 'status': 'success'})
            all_links.extend(links)
        except Exception as e:
            print(f'      Error: {str(e)[:40]}')
            crawled_pages.append({'url': url, 'status': 'failed', 'error': str(e)})
    seen = set()
    unique_links = [l for l in all_links if l not in seen and not seen.add(l)]
    return unique_links, crawled_pages

def crawl_multi_level(cfg, options):
    url = cfg.get('url')
    levels = cfg.get('levels', [])
    delay = options.get('delay_between_requests', 1)
    exts = cfg.get('file_extensions', ['.pdf'])
    max_files = options.get('max_files', 0)
    result = {'url': url, 'levels': [], 'files': [], 'status': 'pending'}
    try:
        current_urls = [url]
        for level_idx, level_config in enumerate(levels):
            level_name = level_config.get('name', f'Level {level_idx + 1}')
            is_download = level_config.get('is_download', False)
            print(f'\n  Level {level_idx + 1}: {level_name} ({len(current_urls)} urls)')
            next_urls, crawled_pages = crawl_level(current_urls, level_config, url, delay, exts)
            result['levels'].append({'name': level_name, 'pages_crawled': len(crawled_pages), 'links_found': len(next_urls)})
            print(f'  Total: {len(next_urls)} links')
            if is_download:
                if max_files > 0: next_urls = next_urls[:max_files]; print(f'  Limited to {max_files} files')
                print(f'\n  Downloading {len(next_urls)} files...')
                file_links = [{'url': u, 'filename': get_filename(u)} for u in next_urls]
                with ThreadPoolExecutor(5) as ex:
                    futures = {ex.submit(download, f['url'], DOWNLOAD_DIR, f['filename']): f for f in file_links}
                    for fut in as_completed(futures):
                        dl = fut.result()
                        result['files'].append(dl)
                        status = 'OK' if dl['status'] == 'success' else 'FAIL'
                        print(f'    [{status}] {dl["filename"][:50]}')
                break
            else:
                current_urls = next_urls
                if not current_urls: print('  No more URLs'); break
        result['status'] = 'success'
    except Exception as e:
        result['status'] = 'failed'
        result['error'] = str(e)
    return result

def crawl_two_level(cfg, options):
    levels = [
        {'name': 'Detail pages', 'selector': cfg.get('level1_selector'), 'max_pages': cfg.get('max_detail_pages', 50)},
        {'name': 'Download links', 'selector': cfg.get('level2_selector'), 'is_download': True}
    ]
    cfg['levels'] = levels
    return crawl_multi_level(cfg, options)

def crawl_one_level(cfg, options):
    url = cfg.get('url')
    exts = cfg.get('file_extensions', ['.pdf'])
    selector = cfg.get('region_selector') or cfg.get('level2_selector')
    max_files = options.get('max_files', 0)
    result = {'url': url, 'files': [], 'status': 'pending'}
    try:
        r = requests.get(url, headers=HEADERS, timeout=30)
        r.raise_for_status()
        links = extract_links(r.text, url, selector, exts, require_ext=not selector)
        if max_files > 0: links = links[:max_files]; print(f'  Limited to {max_files} files')
        print(f'  Found: {len(links)} files')
        file_links = [{'url': u, 'filename': get_filename(u)} for u in links]
        with ThreadPoolExecutor(5) as ex:
            futures = {ex.submit(download, f['url'], DOWNLOAD_DIR, f['filename']): f for f in file_links}
            for fut in as_completed(futures):
                result['files'].append(fut.result())
        result['status'] = 'success'
    except Exception as e:
        result['status'] = 'failed'
        result['error'] = str(e)
    return result

In [15]:
%cd

/root


In [16]:
with open(INPUT_FILE, 'r', encoding='utf-8') as f:
    data = json.load(f)
urls = data.get('urls', [])
options = data.get('options', {})
all_results = []
print(f'URLs: {len(urls)}')
if options.get('max_files'): print(f'Max files: {options["max_files"]}')
print('='*50)
for i, cfg in enumerate(urls, 1):
    url = cfg.get('url')
    mode = cfg.get('crawl_mode', 'one_level')
    print(f'\n[{i}] {url[:50]}...')
    print(f'  Mode: {mode}')
    if mode == 'multi_level': result = crawl_multi_level(cfg, options)
    elif mode == 'two_level': result = crawl_two_level(cfg, options)
    else: result = crawl_one_level(cfg, options)
    ok = sum(1 for f in result['files'] if f['status'] == 'success')
    print(f'\n  Downloaded: {ok}/{len(result["files"])}')
    all_results.append(result)
print('\n' + '='*50)
print('Done!')

FileNotFoundError: [Errno 2] No such file or directory: '/content/drive/MyDrive/Crawl_Data/CrawlData/ssr_urls.json'

In [ ]:
output = {'results': all_results, 'summary': {'urls': len(all_results), 'total_files': sum(len(r['files']) for r in all_results), 'downloaded': sum(sum(1 for f in r['files'] if f['status']=='success') for r in all_results)}, 'crawled_at': datetime.now().isoformat()}
with open(OUTPUT_FILE, 'w', encoding='utf-8') as f:
    json.dump(output, f, indent=2, ensure_ascii=False)
print(f'Total: {output["summary"]["total_files"]}')
print(f'Downloaded: {output["summary"]["downloaded"]}')
print(f'Saved: {OUTPUT_FILE}')